# Lesson 12 - How to Reduce Chat History Wit Agent Scratchpad

Dis notebook dey show how to manage context for long convo wey you dey use Microsoft Agent Framework. As convo long, di token number go dey increase — e fit pass di model context window. We go solve dis with **context summarization pattern** and **agent scratchpad** wey go keep memory steady.

## Wetin You Go Learn:
1. **Why Context Management Dey Important**: To sabi token limit and context window
2. **Context-Aware Agents**: How to build agent wey fit manage im own convo context
3. **Context Summarization Pattern**: How to use tools to short the convo history
4. **Agent Scratchpad**: Memory wey no dey lost even if context reduce

## Wetin You Must Get Before:
- Azure OpenAI setup wit environment variables don set
- Understand basic agent tin from before lessons


## Di Setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Why Context Management Dey Important

Every LLM get limited **context window** — na di maximum tokens wey e fit process for one request. As conversation wey get many turns dey go:

- **Token count dey increase linearly** with every user message and assistant reply.
- **Prompt tokens dey cost pass** because whole history dey sent again every turn.
- For finish, conversation fit **pass context window** and model go either cut am short or give error.

### Ways To Manage Context

| Strategy | How E Dey Work | Trade-off |
|---|---|---|
| **Truncation** | Comot oldest messages | E go lose early context |
| **Summarization** | Make older messages short summary | Some detail go lost, but main points go remain |
| **Scratchpad / External Memory** | Put key facts outside conversation | E go need tool calls, but e fit survive any reduction |

For this notebook we join **summarization** plus **scratchpad tool** so agent fit hold continuity even if conversation history short.


## Di Process for Create One Agent Wey Dey Sabi Di Context


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simuleit Long Tok Tok

Make we waka waka for one kain long tok tok wey get many stops to see as context dey gather. Di agent go suppose hold key tins (di tins wey person like, money wey person get, date wey person wan travel) for all di stops and show say e dey continue. 


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Notice how di agent dey keep di context from earlier tok — e sabi about Japan, sushi, temples, photography, di $3000 budget, solo travel, and di April timeframe. For short konversation dis dey work well, but as di konversation dey grow di full history go dey expensive to re-send.

Make we continue di konversation with more tok to see how context dey accumulate:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

As di tori deep, we fit use **summarization tool** to make di koko gist small and tight. Di agent go call dis tool to save di main palava so even if old message dem drop, di important tin dem go still dey.

Dis kain pattern na di foundation for better way to reduce history:
1. Di agent go find important tin for di tori
2. E go call di summarization tool to save am
3. Old message dem fit comot well-well because di summary dey hold di tins wey matter

Below we define a `summarize_preferences` tool wey di agent fit call to save small small summary of wetin e don sabi.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Summary

For dis lesson, you don learn how to manage context for long-time agent talks using Microsoft Agent Framework:

### Key Concepts
- **Context windows get limit** — every token for conversation history dey cost money and e dey count towards di limit.
- **Summarization tools** dey help agent condense wetin dem don collect for context into small small summaries, e dey reduce token use but e still keep di important info.
- **Agent scratchpads** dey provide memory wey no vanish, e go last even if conversation reduce.

### Wetin You Build
- One **context-aware agent** wey dey keep things steady through multi-turn conversations
- One **summarization tool** (`summarize_preferences`) wey go record important user detail for small small format
- One **multi-turn conversation** wey fit show how context fit last and change matter

### Real-World Applications
- **Customer Service Bots**: Dem fit remember wetin customer like for long support session
- **Personal Assistants**: Dem fit follow current projects without make person explain context again
- **Educational Tutors**: Dem fit keep track of student progress for plenty interactions

### Next Steps
- Make full scratchpad tool wey fit save files steady
- Add automatic truncation for history after summarization
- Combine am with vector databases for semantic memory search
- Build agents wey fit resume talks days after, with full context


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
